# Exam Scheduling — Experiment Notebook

12 C++ algorithms + Auto-Tuner. Falls back to Python automatically if C++ isn't compiled.

**Algorithms:** Greedy, Tabu Search, Kempe Chain, SA, ALNS, Great Deluge, ABC, GA, LAHC, WOA, CP-SAT, GVNS

**Workflow:** Config &rarr; Run experiments &rarr; Tables & plots &rarr; Export | Auto-tune &rarr; Chain discovery

**SideNotes:** Make sure to run the main cells before experiment to prevent errors. 

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))

import importlib
import pandas as pd
from IPython.display import display, HTML

from core.parser import parse_itc2007_exam
from core.generator import generate_synthetic, write_itc2007_format
from algorithms.cpp_bridge import run_solver
from utils.results_logger import ResultsLogger
from utils.batch_manager import BatchManager

# Force-reload plotting so re-running this cell picks up any changes
import utils.plotting
importlib.reload(utils.plotting)
from utils.plotting import *
import matplotlib.pyplot as plt

try:
    from algorithms.ip_solver import solve_ip
    HAS_IP = True
except ImportError:
    HAS_IP = False

bm = BatchManager()
print("Ready. All solvers route through C++ binary.")

Ready. All solvers route through C++ binary.


## Batch Management

Each experiment run is isolated in its own directory under `results/`.  
- **Auto mode**: Run the next cell as-is to create a timestamped batch.  
- **Manual mode**: Set `BATCH_NAME` to give it a custom name.  
- **Load previous**: Set `BATCH_MODE = "load"` and `BATCH_LOAD = "baseline"` (or batch ID/name).

All results, logs, plots, and solution files go into the active batch directory.

In [2]:
# ── Batch setup ──────────────────────────────────────────
# BATCH_MODE: "new" (create new batch) or "load" (resume/view previous)
BATCH_MODE = "load"

# For "new" mode: set a name, or None for auto-timestamped
BATCH_NAME = "tunning8"          # e.g. "tuning_sa", "v14_optimized"

# For "load" mode: batch ID (int), dirname, or partial name match
BATCH_LOAD = "batch_017_tunning8"    # e.g. 1, "batch_001_baseline", "baseline"

# ─────────────────────────────────────────────────────────
if BATCH_MODE == "load":
    bm.load_batch(BATCH_LOAD)
else:
    bm.new_batch(BATCH_NAME)

logger = bm.logger
print(f"\nActive batch: {os.path.basename(bm.active_dir)}")
print(f"Output dir:   {bm.active_dir}")
print(f"Existing records: {len(logger.load_all())}")

[Batch] Loaded: batch_017_tunning8

Active batch: batch_017_tunning8
Output dir:   results/batch_017_tunning8
Existing records: 97


In [ ]:
# ── List all batches ─────────────────────────────────────
bm.print_batches()

## Config

In [3]:
from tooling.tuned_params import load_params_flat

DATASETS = {
    "exam_comp_set1": "instances/exam_comp_set1.exam",
    "exam_comp_set2": "instances/exam_comp_set2.exam",
    "exam_comp_set3": "instances/exam_comp_set3.exam",
    "exam_comp_set4": "instances/exam_comp_set4.exam",
    "exam_comp_set5": "instances/exam_comp_set5.exam",
    "exam_comp_set6": "instances/exam_comp_set6.exam",
    "exam_comp_set7": "instances/exam_comp_set7.exam",
    "exam_comp_set8": "instances/exam_comp_set8.exam"
}

# Algorithm: single name, comma-separated list, or "all"
# Examples: "sa", "sa,gd,tabu", "all"
# "all" runs all 12 individual algorithms.
ALGO = "all"

# ── Experiment mode ──────────────────────────────────
# "individual" → runs each algorithm separately using ALGO above
# "chain"      → runs a warm-started chain using EXPERIMENT_CHAIN below
EXPERIMENT_MODE = "individual"

# Chain used when EXPERIMENT_MODE = "chain".
# None → auto-load the tuned best chain from tuned_params.json (falls back to error).
# Override by providing a list of (algo, params_dict) tuples.
EXPERIMENT_CHAIN = None
# EXPERIMENT_CHAIN = [("sa", {"sa_iters": 5000}), ("gd", {"gd_iters": 5000})]

# Load tuned defaults. load_params_flat() always returns every key — either from
# tooling/tuned_params.json or FALLBACK_PARAMS in tooling/tuned_params.py.
_gp = load_params_flat()

TABU_ITERS     = _gp['tabu_iters']
TABU_TENURE    = _gp['tabu_tenure']
TABU_PATIENCE  = _gp['tabu_patience']
SA_ITERS       = _gp['sa_iters']
KEMPE_ITERS    = _gp['kempe_iters']
ALNS_ITERS     = _gp['alns_iters']
GD_ITERS       = _gp['gd_iters']
ABC_POP        = _gp['abc_pop']
ABC_ITERS      = _gp['abc_iters']
GA_POP         = _gp['ga_pop']
GA_ITERS       = _gp['ga_iters']
LAHC_ITERS     = _gp['lahc_iters']
LAHC_LIST      = _gp['lahc_list']
WOA_POP        = _gp['woa_pop']
WOA_ITERS      = _gp['woa_iters']
CPSAT_TIME     = _gp['cpsat_time']
VNS_ITERS      = _gp['vns_iters']
VNS_BUDGET     = _gp['vns_budget']
SEED           = 42

NUM_TRIALS     = 1       # Num of trials

# IP solver (Python, only for small instances)
RUN_IP         = False
IP_TIME_LIMIT  = 120

VERBOSE = True

print(f"Params source: {'tooling/tuned_params.json' if os.path.isfile('tooling/tuned_params.json') else 'hardcoded fallback'}")

Params source: tooling/tuned_params.json


## Load Datasets

In [4]:
problems = {}
for name, path in DATASETS.items():
    if os.path.isfile(path):
        problems[name] = parse_itc2007_exam(path)
        p = problems[name]
        print(f"{name}: {p.num_exams()} exams, {p.num_periods()} periods, "
              f"{p.num_rooms()} rooms, {p.num_students()} students, "
              f"density={p.conflict_density():.3f}")
    else:
        print(f"SKIP: {path} not found")
print(f"\nLoaded {len(problems)} dataset(s)")

exam_comp_set1: 607 exams, 54 periods, 7 rooms, 7883 students, density=0.050
exam_comp_set2: 870 exams, 40 periods, 49 rooms, 12484 students, density=0.012
exam_comp_set3: 934 exams, 36 periods, 48 rooms, 16365 students, density=0.026
exam_comp_set4: 273 exams, 21 periods, 1 rooms, 4421 students, density=0.150
exam_comp_set5: 1018 exams, 42 periods, 3 rooms, 8719 students, density=0.009
exam_comp_set6: 242 exams, 16 periods, 8 rooms, 7909 students, density=0.062
exam_comp_set7: 1096 exams, 80 periods, 15 rooms, 13795 students, density=0.019
exam_comp_set8: 598 exams, 80 periods, 8 rooms, 7718 students, density=0.045

Loaded 8 dataset(s)


## Run Experiments

In [ ]:
session_records = []

if EXPERIMENT_MODE == "chain":
    from tooling.tuned_params import load_best_chain
    from algorithms.cpp_bridge import run_chain as _run_chain

    chain = EXPERIMENT_CHAIN or load_best_chain()
    if not chain:
        raise RuntimeError(
            "EXPERIMENT_MODE='chain' but no chain provided and no tuned best_chain "
            "found in tuned_params.json. Set EXPERIMENT_CHAIN manually or run the "
            "auto-tuner first."
        )
    chain_label = "Chain(" + "\u2192".join(s[0] for s in chain) + ")"
    print(f"Chain mode \u2014 {chain_label}")

    for ds_name, ds_path in DATASETS.items():
        if ds_name not in problems:
            continue
        problem = problems[ds_name]
        print(f"\n{'='*60}\n  {ds_name} ({problem.num_exams()} exams)\n{'='*60}")

        for trial in range(NUM_TRIALS):
            seed = SEED + trial * 1000
            result = _run_chain(ds_path, chain, seed=seed)
            if result is None:
                print(f"  [{chain_label}] t={trial} FAILED")
                continue
            config = {"chain": [(a, dict(p)) for a, p in chain], "seed": seed}
            rec = logger.log_run(ds_name, problem, result, config=config, trial=trial)
            session_records.append(rec)
            ev = result['evaluation']
            feasible = ev.hard == 0
            print(f"  [{chain_label:<22}] t={trial}  feasible={feasible}  "
                  f"hard={ev.hard}  soft={ev.soft:>8}  time={result['runtime']:.3f}s  "
                  f"mem={result['memory_peak_mb']:.1f}MB")

else:  # individual mode
    for ds_name, ds_path in DATASETS.items():
        if ds_name not in problems:
            continue
        problem = problems[ds_name]
        print(f"\n{'='*60}\n  {ds_name} ({problem.num_exams()} exams)\n{'='*60}")

        for trial in range(NUM_TRIALS):
            seed = SEED + trial * 1000

            results = run_solver(
                ds_path, algo=ALGO,
                tabu_iters=TABU_ITERS, tabu_tenure=TABU_TENURE,
                tabu_patience=TABU_PATIENCE,
                sa_iters=SA_ITERS, kempe_iters=KEMPE_ITERS,
                alns_iters=ALNS_ITERS, gd_iters=GD_ITERS,
                abc_pop=ABC_POP, abc_iters=ABC_ITERS,
                ga_pop=GA_POP, ga_iters=GA_ITERS,
                lahc_iters=LAHC_ITERS, lahc_list=LAHC_LIST,
                woa_pop=WOA_POP, woa_iters=WOA_ITERS,
                cpsat_time=CPSAT_TIME,
                vns_iters=VNS_ITERS, vns_budget=VNS_BUDGET,
                seed=seed, verbose=VERBOSE,
            )

            for algo_name, r in results.items():
                if algo_name == 'Greedy' and trial > 0:
                    continue

                config = {
                    "tabu_iters": TABU_ITERS, "tabu_tenure": TABU_TENURE,
                    "sa_iters": SA_ITERS, "kempe_iters": KEMPE_ITERS,
                    "alns_iters": ALNS_ITERS, "gd_iters": GD_ITERS,
                    "abc_pop": ABC_POP, "abc_iters": ABC_ITERS,
                    "ga_pop": GA_POP, "ga_iters": GA_ITERS,
                    "lahc_iters": LAHC_ITERS, "lahc_list": LAHC_LIST,
                    "woa_pop": WOA_POP, "woa_iters": WOA_ITERS,
                    "cpsat_time": CPSAT_TIME,
                    "vns_iters": VNS_ITERS, "vns_budget": VNS_BUDGET,
                    "seed": seed,
                }
                rec = logger.log_run(ds_name, problem, r, config=config, trial=trial)
                session_records.append(rec)
                ev = r['evaluation']
                feasible = ev.feasible if hasattr(ev, 'feasible') else ev.is_feasible
                soft = ev.soft if hasattr(ev, 'soft') else ev.soft_penalty
                hard = ev.hard if hasattr(ev, 'hard') else ev.hard_violations
                print(f"  [{algo_name:<22}] t={trial}  feasible={feasible}  "
                      f"hard={hard}  soft={soft:>8}  time={r['runtime']:.3f}s")

            if RUN_IP and HAS_IP and problem.num_exams() <= 300:
                r = solve_ip(problem, time_limit=IP_TIME_LIMIT, verbose=VERBOSE)
                rec = logger.log_run(ds_name, problem, r,
                    config={"ip_time_limit": IP_TIME_LIMIT}, trial=trial)
                session_records.append(rec)

print(f"\nLogged {len(session_records)} records this session")
print(f"Total in log: {len(logger.load_all())}")


## Results Table

In [5]:
df = logger.to_dataframe()

summary = df.groupby(['dataset', 'algorithm']).agg(
    runs=('feasible', 'count'),
    feasible_pct=('feasible', lambda x: f"{x.mean()*100:.0f}%"),
    hard_mean=('hard_violations', 'mean'),
    soft_mean=('soft_penalty', 'mean'),
    soft_std=('soft_penalty', 'std'),
    soft_min=('soft_penalty', 'min'),
    runtime_mean=('runtime', 'mean'),
).round(2).reset_index()

display(HTML("<h3>All Logged Runs</h3>"))
display(summary)


,dataset,algorithm,runs,feasible_pct,hard_mean,soft_mean,soft_std,soft_min,runtime_mean
0,exam_comp_set1,ABC,1,100%,0.0,13609.0,NaN,13609,27.97
1,exam_comp_set1,ALNS,1,100%,0.0,11209.0,NaN,11209,11.35
2,exam_comp_set1,CP-SAT B&B,1,100%,0.0,14181.0,NaN,14181,1.50
3,exam_comp_set1,GVNS,1,100%,0.0,11317.0,NaN,11317,6.83
4,exam_comp_set1,Genetic Algorithm,1,100%,0.0,21126.0,NaN,21126,5.80
...,...,...,...,...,...,...,...,...,...
92,exam_comp_set8,Kempe Chain,1,100%,0.0,14435.0,NaN,14435,7.44
93,exam_comp_set8,LAHC,1,100%,0.0,15036.0,NaN,15036,1.22
94,exam_comp_set8,Multi-Neighbourhood SA,1,100%,0.0,19400.0,NaN,19400,1.10
95,exam_comp_set8,Tabu Search,1,100%,0.0,12861.0,NaN,12861,9.07


## Aggregated Results & Export

In [ ]:
agg_path = logger.save_aggregated()
csv_path = logger.to_csv()
print(f"Aggregated JSON: {agg_path}")
print(f"Full CSV: {csv_path}")

agg_df = logger.aggregate_to_dataframe()
cols = ['algorithm','dataset','count','feasible_rate',
        'runtime_mean','runtime_std','soft_penalty_mean','soft_penalty_std','soft_penalty_min']
display(agg_df[[c for c in cols if c in agg_df.columns]].round(2))

## Experiment Summary

Interactive line chart — soft penalty and runtime across all datasets, averaged over seeds. Click legend entries to toggle algorithms.

In [6]:
import importlib, utils.plotting as _pl
importlib.reload(_pl)
from utils.plotting import (plot_experiment_summary, plot_algo_bars,
                             plot_algo_boxes, plot_algo_radar,
                             plot_algo_scatter, plot_algo_heatmap,
                             save_all_plotly)

plot_experiment_summary(df)   # line: soft + runtime + memory across datasets
plot_algo_bars(df)            # horizontal bars: soft + runtime + memory
plot_algo_boxes(df)           # box plot: soft penalty distribution
plot_algo_radar(df)           # radar: normalised performance profile
plot_algo_scatter(df)         # scatter: quality vs runtime
plot_algo_heatmap(df);        # heatmap: algorithm x dataset soft penalty

save_all_plotly(df)         # uncomment to save all charts as PNG


[Plot] Saved 6 charts to graphs/


## Parameter Sweep

Runs a single algorithm with increasing iteration counts to study convergence.
Change `SWEEP_ALGO` to sweep any algorithm.

In [ ]:

SWEEP_ALGO    = "tabu"          # "tabu","sa","kempe","alns","gd","abc","ga","lahc","woa","vns"
SWEEP_PARAM   = "tabu_iters"    # kwarg name passed to run_solver
SWEEP_DATASET = list(DATASETS.keys())[0]
SWEEP_PATH    = DATASETS[SWEEP_DATASET]
PARAM_VALUES  = [100, 250, 500, 1000, 2000, 3000]
SWEEP_TRIALS  = 3

sweep_records = []
problem = problems[SWEEP_DATASET]

for val in PARAM_VALUES:
    for trial in range(SWEEP_TRIALS):
        seed = 42 + trial * 1000
        results = run_solver(
            SWEEP_PATH, algo=SWEEP_ALGO,
            seed=seed, verbose=False,
            **{SWEEP_PARAM: val},
        )
        for name, r in results.items():
            rec = logger.log_run(SWEEP_DATASET, problem, r,
                config={SWEEP_PARAM: val, "seed": seed},
                trial=trial, notes=f"sweep_{SWEEP_PARAM}_{val}")
            sweep_records.append(rec)
        print(f"  {SWEEP_PARAM}={val} t={trial}: soft={rec['soft_penalty']} time={rec['runtime']:.2f}s")

sdf = pd.DataFrame(sweep_records)
sdf['sweep_val'] = sdf['config'].apply(lambda c: c.get(SWEEP_PARAM, 0))
plot_parameter_sensitivity(sdf, 'sweep_val', metric='soft_penalty', algorithm=sweep_records[0]['algorithm'])
plt.xlabel(SWEEP_PARAM)
plt.show()


## Generate Synthetic Instances

Generates competition-level instances at varying sizes using presets:
`"easy"`, `"medium"`, `"hard"`, `"competition"`.

These can be used by the auto-tuner or run experiments separately.

In [ ]:
SYNTH_CONFIGS = [(342, "competition"), (675, "competition"), (879, "competition")]

os.makedirs("instances", exist_ok=True)
generated = []

for n, preset in SYNTH_CONFIGS:
    exam_path = f"instances/synthetic_{preset}_{n}.exam"
    if not os.path.isfile(exam_path):
        problem = generate_synthetic(num_exams=n, preset=preset, seed=42 + n)
        write_itc2007_format(problem, exam_path)
        tag = "generated"
    else:
        problem = parse_itc2007_exam(exam_path)
        tag = "exists"
    generated.append((n, preset, exam_path, problem))
    print(f"  [{tag}] {exam_path}: {problem.num_exams()} exams, "
          f"{problem.num_periods()} periods, {problem.num_rooms()} rooms, "
          f"density={problem.conflict_density():.3f}")

print(f"\n{len(generated)} synthetic datasets ready in instances/")


## Auto-Tuner

Automated parameter optimization + algorithm chain discovery. Uses a **chain-first** strategy, discovers winning chains before tuning, so compute is spent only on algorithms that matter in combination.

1. **Quick Screen** — all algorithms with defaults on ALL datasets (parallel)
2. **Chain Discovery** — natural-selection tournament over warm-started chains using full algo pool + ITC-literature combos (sa→gd, kempe→sa, tabu→sa→gd, etc.)
3. **Extract Core Algos** — frequency analysis across top chains picks the 2–4 algorithms that actually drive results
4. **Deep Parameter Tuning** — random + perturbation sampling on core algos only (2× trial depth since fewer algos)
5. **Chain Rediscovery** — re-runs chain tournament with tuned params on the focused pool
6. **Final Validation** — multi-seed robustness check on ALL datasets

**Why chain-first?** SA standalone=16.2 (bad), but ITC 2007 winners used SA+Kempe combos. GD standalone=104.3 (terrible), but Müller's 1st-place used GD as finisher. Standalone scores are misleading — synergy matters more. Chain-first lets synergy decide which algos to tune.

**Global mode** evaluates across multiple datasets using geometric mean of normalized scores, prevents overfitting to any single dataset. Wall-time budget ensures it doesn't run forever.

**Auto-update params:** After tuning, if the new params beat the current golden defaults (better aggregate score, comparable trial count, no single-dataset regression >15%), they're automatically saved to `tooling/tuned_params.json`. All consumers (CLI, notebook, cpp_bridge) pick up the new defaults on next run. Every update is logged to `tooling/tuned_params_log.json` for rollback.

```bash
# CLI
python -m tooling.auto_tuner --all-sets
# Flags
--synthetic       # + synthetic dataset
--max-time 20     # 20 min budget
--no-auto-update  # tune without updating defaults

# Param management
python main.py --show-params        # view active defaults + history
python main.py --rollback-params 2  # restore version 2
```

In [ ]:
import glob as globmod
import json
from tooling import auto_tuner as _at_mod
importlib.reload(_at_mod)
AutoTuner = _at_mod.AutoTuner
generate_synthetic_dataset = _at_mod.generate_synthetic_dataset
from tooling.tuned_params import load_metadata, list_versions

# ── Global mode: all ITC 2007 sets + synthetic ──────────
TUNE_DATASETS  = sorted(globmod.glob("instances/exam_comp_set*.exam"))
TUNE_OUTPUT    = os.path.join(bm.active_dir, "tuning")
TUNE_WORKERS   = 5       # parallel workers
PARAM_TRIALS   = 3       # trials per algo
CHAIN_POP      = 4       # top survivors mutate quickly
CHAIN_ROUNDS   = 2       # tournament rounds (plateau detection stops early anyway)
MAX_TIME       = 2400    # 30 min wall-time budget
EVAL_DATASETS  = 2       # eval on 2 representative datasets per trial (small+large)
AUTO_UPDATE    = True    # auto-update golden params if improved

# ── Synthetic: scale difficulty with size ────────────────
# Small instances can't handle competition density (too few room×period slots)
INCLUDE_SYNTH  = True
SYNTH_CONFIGS  = [(342, "competition"), (675, "competition"), (879, "competition"),
                  (569, "competition"), (710, "competition")]

if INCLUDE_SYNTH:
    synth_dir = os.path.join(TUNE_OUTPUT, "_synthetic")
    for size, preset in SYNTH_CONFIGS:
        syn_path = generate_synthetic_dataset(
            synth_dir, num_exams=size, preset=preset,
            seed=SEED + size)
        TUNE_DATASETS.append(syn_path)

print(f"Tuning on {len(TUNE_DATASETS)} datasets:")
for d in TUNE_DATASETS:
    print(f"  {os.path.basename(d)}")

tuner = AutoTuner(
    datasets=TUNE_DATASETS,
    output_dir=TUNE_OUTPUT,
    max_workers=TUNE_WORKERS,
    param_trials=PARAM_TRIALS,
    chain_pop=CHAIN_POP,
    chain_rounds=CHAIN_ROUNDS,
    max_time=MAX_TIME,
    eval_datasets=EVAL_DATASETS,
    seed=SEED,
    auto_update=AUTO_UPDATE,
)
tuner.run()

# ── Report ────────────────────────────────────────────
report_path = os.path.join(TUNE_OUTPUT, "tuning_report.json")
if os.path.isfile(report_path):
    report = json.load(open(report_path))
    score = report['best_score']
    fmt = f"{score:.4f}" if score < 100 else f"{score:.0f}"
    print(f"\nBest score: {fmt}")
    print(f"Best config: {report['best_config']}")
else:
    print("\nNo report generated (may have timed out before final phase)")

# ── Golden params status ────────────────────────
meta = load_metadata()
if meta:
    print(f"\nActive golden params: v{meta.get('version', '?')} "
          f"(score={meta.get('aggregate_score', '?')}, "
          f"{meta.get('source', '?')})")
versions = list_versions()
if versions:
    print(f"Version history: {len(versions)} entries")
    for v, ts, sc, src in versions[-3:]:
        print(f"  v{v} score={sc} ({src})")


## Synthetic Size Scan — Best Chain

Runs the tuned global-best chain on progressively larger synthetic instances
(25 → 200 exams, step 25) to see how soft penalty, runtime, and peak memory
scale with problem size. Uses 3 seeds per size for error ribbons.

Toggle `USE_MANUAL_CHAIN = True` to override the tuned chain with the
`MANUAL_CHAIN` literal below — useful when the auto-tuner hasn't been run yet
or when you want to compare a specific chain.


In [ ]:
from tooling.tuned_params import load_best_chain

# ── Chain selection ────────────────────────────────────
USE_MANUAL_CHAIN = False
MANUAL_CHAIN = [
    ("sa", {"sa_iters": 5000}),
    ("kempe", {"kempe_iters": 15832}),
    ("gd", {"gd_iters": 5000}),
]

_tuned = load_best_chain()
if USE_MANUAL_CHAIN:
    chain, source = MANUAL_CHAIN, "manual (forced)"
elif _tuned:
    chain, source = _tuned, "tuned_params.json"
else:
    chain, source = MANUAL_CHAIN, "manual (fallback — no tuned chain found)"

chain_label = " → ".join(step[0] for step in chain)
print(f"Chain: {chain_label}  (source: {source})")

# ── Scan config ────────────────────────────────────────
SCAN_SIZES  = [25, 50, 75, 100, 125, 150, 175, 200]
SCAN_SEEDS  = [42, 123, 789]
SCAN_PRESET = "competition"


In [ ]:
from algorithms.cpp_bridge import run_chain
from utils.plotting import plot_continuous_scan
import matplotlib.pyplot as plt

scan_records = []
for n in SCAN_SIZES:
    exam_path = f"instances/synthetic_scan_{n}.exam"
    if not os.path.isfile(exam_path):
        problem = generate_synthetic(num_exams=n, preset=SCAN_PRESET, seed=42 + n)
        write_itc2007_format(problem, exam_path)
    for seed in SCAN_SEEDS:
        result = run_chain(exam_path, chain, seed=seed)
        if result is None:
            print(f"  n={n} seed={seed}: FAILED")
            continue
        scan_records.append({
            "num_exams": n,
            "seed": seed,
            "soft_penalty": result["soft_penalty"],
            "runtime": result["total_runtime"],
            "memory_peak_mb": result["memory_peak_mb"],
        })
    done = len([r for r in scan_records if r["num_exams"] == n])
    print(f"  n={n}: {done}/{len(SCAN_SEEDS)} seeds done")

scan_df = pd.DataFrame(scan_records)
display(scan_df.groupby("num_exams").agg({
    "soft_penalty":   ["mean", "std"],
    "runtime":        ["mean", "std"],
    "memory_peak_mb": ["mean", "std"],
}).round(2))

plot_continuous_scan(scan_df, title=f"Synthetic Scan — {chain_label}")
plt.show()


## Reset & Utilities

**Log stats** — shows how many records and file size for the active batch.  
**Quick Reset** — clears the active batch's log so you can re-run.  
**Switch batch** — change active batch without re-running imports.

In [ ]:
# ─── Log stats (active batch) ───
print(f"Active batch: {os.path.basename(bm.active_dir)}")
if os.path.isfile(logger.filepath):
    n = len(logger.load_all())
    kb = os.path.getsize(logger.filepath) / 1024
    print(f"Log: {n} records, {kb:.1f} KB")
else:
    print("No log file found.")
print()
bm.print_batches()

In [ ]:
# === QUICK RESET — clears the ACTIVE BATCH's log ===
"""
logger.clear()
session_records.clear()
if 'df' in dir(): del df
print(f"Cleared log for: {os.path.basename(bm.active_dir)}")
print("Re-run experiments to repopulate.")
"""

In [ ]:
# === Switch active batch without re-running imports ===
"""
bm.load_batch("baseline")   # ID, dirname, or partial name
logger = bm.logger
print(f"Switched to: {os.path.basename(bm.active_dir)}")
print(f"Records: {len(logger.load_all())}")
"""

In [ ]:
# === Clear PNG files from result batches ===
# Usage:
#   clear_batch_pngs("baseline")                    # match a batch by partial name/ID
#   clear_batch_pngs(bm.active_dir)                 # the active batch (full path also OK)
#   clear_batch_pngs(all_batches=True)              # every batch_*/ dir
#   clear_batch_pngs(all_batches=True, include_best=True)  # also clear results/best/*
#   clear_batch_pngs(all_batches=True, dry_run=True)       # preview, don't delete
def clear_batch_pngs(batch=None, all_batches=False, include_best=False, dry_run=False):
    import glob as _glob
    results_root = "results"
    targets = []

    if all_batches:
        targets.extend(sorted(_glob.glob(f"{results_root}/batch_*")))
        if include_best:
            targets.extend(sorted(_glob.glob(f"{results_root}/best/batch_*")))
    elif batch is not None:
        b = str(batch)
        # Allow full paths, dirnames, partial names, or numeric IDs
        if os.path.isdir(b):
            targets.append(b)
        else:
            hits = sorted(_glob.glob(f"{results_root}/batch_*{b}*"))
            if not hits and b.isdigit():
                hits = sorted(_glob.glob(f"{results_root}/batch_{int(b):03d}_*"))
            if not hits:
                print(f"No batch matching: {batch!r}")
                return
            targets.extend(hits)
    else:
        print("Provide batch=... or all_batches=True")
        return

    total_removed = 0
    for d in targets:
        pngs = _glob.glob(f"{d}/*.png")
        if not pngs:
            continue
        label = os.path.basename(d)
        if dry_run:
            print(f"  [dry] {label}: would remove {len(pngs)} png(s)")
        else:
            for p in pngs:
                try: os.remove(p)
                except OSError: pass
            print(f"  {label}: removed {len(pngs)} png(s)")
        total_removed += len(pngs)
    verb = "Would remove" if dry_run else "Removed"
    print(f"{verb} {total_removed} PNG file(s) across {len(targets)} batch(es)")
